# 🧠 NeuroVision 3D: Deep Learning 3D Brain Tumour Segmentation
### BraTS Dataset Training with PyTorch & MONAI (SegResNet / 3D U-Net)

This notebook trains a state-of-the-art volumetric 3D segmentation network on brain MRI scans.

**Key Architecture & Training Details:**
- **Framework:** PyTorch + MONAI (Medical Open Network for AI)
- **Model:** 3D `SegResNet` (BraTS challenge-winning architecture, memory efficient for 16GB GPUs)
- **Modalities:** 4 coregistered MRI sequences (`FLAIR`, `T1`, `T1ce`, `T2`)
- **Target Compound Classes:**
  - **WT (Whole Tumor):** Necrotic core + Edema + Enhancing tumor (Labels 1 + 2 + 4)
  - **TC (Tumor Core):** Necrotic core + Enhancing tumor (Labels 1 + 4)
  - **ET (Enhancing Tumor):** Active enhancing tumor (Label 4)
- **Training Technique:** Mixed Precision (`torch.cuda.amp`), Patch-based Random Cropping, Sliding-Window Inference
- **Loss Function:** Compound Dice Loss (`sigmoid=True` for multi-label binary evaluation)

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q "monai[all]" nibabel matplotlib tqdm

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Import Libraries & Reproducibility Setup

In [ ]:
import os
import glob
import time
import json
import random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm import tqdm

import monai
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    RandSpatialCropd,
    RandFlipd,
    RandRotate90d,
    MapTransform,
)
from monai.networks.nets import SegResNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import Dataset, DataLoader, decollate_batch

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
monai.utils.set_determinism(seed=SEED)
print("Environment initialized successfully.")

## Step 3: Locate BraTS Dataset on Kaggle
This cell searches standard Kaggle input directories to find BraTS data files.

In [ ]:
def find_brats_data_dicts(kaggle_input_dir="/kaggle/input"):
    """
    Discovers BraTS patient directories and groups the 4 modalities with the segmentation mask.
    Supports BraTS 2020, BraTS 2021, and similar dataset layouts.
    """
    data_dicts = []
    
    # Search for all segmentation files
    seg_files = glob.glob(f"{kaggle_input_dir}/**/*_seg.nii*", recursive=True)
    if not seg_files:
        seg_files = glob.glob(f"{kaggle_input_dir}/**/seg.nii*", recursive=True)
        
    print(f"Found {len(seg_files)} segmentation candidate files.")
    
    for seg_path in seg_files:
        folder = os.path.dirname(seg_path)
        base_prefix = os.path.basename(seg_path).replace("_seg.nii.gz", "").replace("_seg.nii", "")
        
        # Identify the 4 modalities
        flair = glob.glob(f"{folder}/*flair.nii*")
        t1 = glob.glob(f"{folder}/*t1.nii*")
        t1ce = glob.glob(f"{folder}/*t1ce.nii*")
        t2 = glob.glob(f"{folder}/*t2.nii*")
        
        # Remove false matches (e.g. t1 matching t1ce)
        t1 = [f for f in t1 if "t1ce" not in f.lower()]
        
        if flair and t1 and t1ce and t2:
            data_dicts.append({
                "image": [flair[0], t1ce[0], t1[0], t2[0]], # Order: FLAIR, T1ce, T1, T2
                "label": seg_path,
                "subject_id": base_prefix
            })
            
    return data_dicts

all_samples = find_brats_data_dicts()
print(f"Successfully matched {len(all_samples)} complete multi-modal cases.")

# If testing or working with limited time, you can limit the sample size
# e.g., all_samples = all_samples[:80]
if len(all_samples) > 0:
    print("Example Case:", all_samples[0])
else:
    print("⚠️ No files found! Make sure you attached a BraTS dataset in Kaggle (e.g. 'awsaf49/brats20-dataset-training-validation')")

## Step 4: Medical Preprocessing & Multi-Channel BraTS Label Mapping

BraTS annotations use:
- **Label 1:** Necrotic Core & Non-enhancing tumor
- **Label 2:** Peritumoral Edema
- **Label 4:** Enhancing Tumor

We convert these into **3 independent binary channels** for robust training:
1. **Whole Tumor (WT):** `label == 1 | label == 2 | label == 4`
2. **Tumor Core (TC):** `label == 1 | label == 4`
3. **Enhancing Tumor (ET):** `label == 4`

In [ ]:
class ConvertToBraTSMultiChanneld(MapTransform):
    """
    Custom transform to convert discrete BraTS labels into 3 overlapping target channels:
    Channel 0: Whole Tumor (WT) -> Labels 1, 2, 4
    Channel 1: Tumor Core (TC)  -> Labels 1, 4
    Channel 2: Enhancing Tumor (ET) -> Label 4
    """
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            label = d[key]
            # Whole Tumor
            result.append(torch.logical_or(torch.logical_or(label == 1, label == 2), label == 4))
            # Tumor Core
            result.append(torch.logical_or(label == 1, label == 4))
            # Enhancing Tumor
            result.append(label == 4)
            d[key] = torch.stack(result, axis=0).float()
        return d

# Define Training Transforms
ROI_SIZE = (128, 128, 128)  # Patch crop size (balances VRAM and 3D contextual view)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToBraTSMultiChanneld(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "nearest"),
    ),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandSpatialCropd(
        keys=["image", "label"],
        roi_size=ROI_SIZE,
        random_size=False,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
])

# Define Validation Transforms (full volume evaluation)
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToBraTSMultiChanneld(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "nearest"),
    ),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

print("MONAI preprocessing and augmentation pipelines configured.")

## Step 5: Train/Validation Split & DataLoaders

In [ ]:
# Split into 85% Train, 15% Validation (patient-level)
split_idx = int(0.85 * len(all_samples))
train_files = all_samples[:split_idx]
val_files = all_samples[split_idx:]

print(f"Training Patients: {len(train_files)} | Validation Patients: {len(val_files)}")

# Create Datasets and DataLoaders
# Set num_workers=2 for Kaggle environments
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=1,       # 1 volumetric patch per step fits comfortably in 16GB VRAM
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("DataLoaders ready.")

## Step 6: 3D SegResNet Architecture & Compound Loss Setup

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Initialize 3D SegResNet:
# - 4 Input Channels: FLAIR, T1ce, T1, T2
# - 3 Output Channels: WT, TC, ET
model = SegResNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=3,
    init_filters=16,
    blocks_down=[1, 2, 2, 4],
    blocks_up=[1, 1, 1],
    dropout_prob=0.2,
).to(device)

# Loss: Dice Loss with sigmoid activation for multi-label compound outputs
loss_function = DiceLoss(smooth_nr=1e-5, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)

# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Cosine Annealing Learning Rate Scheduler
MAX_EPOCHS = 40  # Set between 30 and 50 for solid convergence on Kaggle
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

# Metric: DiceMetric for evaluation
dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

print(f"Model instantiated on {device}.")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {total_params:,}")

## Step 7: Training Loop with Mixed Precision (`torch.cuda.amp`)

In [ ]:
VAL_INTERVAL = 2  # Run validation every 2 epochs
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
metric_values_wt = []
metric_values_tc = []
metric_values_et = []

scaler = torch.cuda.amp.GradScaler()
output_dir = "/kaggle/working/neurovision_checkpoints"
os.makedirs(output_dir, exist_ok=True)
best_model_path = os.path.join(output_dir, "neurovision_3d_best_model.pth")

print(f"Starting training for {MAX_EPOCHS} epochs...")
start_time = time.time()

for epoch in range(MAX_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{MAX_EPOCHS} ---")
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in tqdm(train_loader, desc=f"Train Epoch {epoch + 1}"):
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
    lr_scheduler.step()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"Epoch {epoch + 1} Average Training Loss: {epoch_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Validation Step
    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        with torch.no_grad():
            for val_data in tqdm(val_loader, desc=f"Val Epoch {epoch + 1}"):
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                
                # 3D Sliding Window Inference to handle whole MRI volume
                val_outputs = sliding_window_inference(
                    inputs=val_inputs,
                    roi_size=ROI_SIZE,
                    sw_batch_size=1,
                    predictor=model,
                    overlap=0.5,
                    mode="gaussian",
                )
                
                # Convert raw logits to binary mask via sigmoid > 0.5
                val_outputs = (torch.sigmoid(val_outputs) > 0.5).float()
                
                dice_metric(y_pred=val_outputs, y=val_labels)
                dice_metric_batch(y_pred=val_outputs, y=val_labels)
                
            # Aggregate validation scores
            mean_dice = dice_metric.aggregate().item()
            dice_metric.reset()
            
            batch_dice = dice_metric_batch.aggregate()
            dice_metric_batch.reset()
            
            wt_dice, tc_dice, et_dice = batch_dice[0].item(), batch_dice[1].item(), batch_dice[2].item()
            
            metric_values.append(mean_dice)
            metric_values_wt.append(wt_dice)
            metric_values_tc.append(tc_dice)
            metric_values_et.append(et_dice)
            
            print(f"Validation Mean Dice: {mean_dice:.4f} [WT: {wt_dice:.4f}, TC: {tc_dice:.4f}, ET: {et_dice:.4f}]")
            
            # Save best checkpoint
            if mean_dice > best_metric:
                best_metric = mean_dice
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), best_model_path)
                print(f"🔥 Saved new best model at epoch {epoch + 1} with Mean Dice: {best_metric:.4f}")

total_time = (time.time() - start_time) / 60
print(f"\nTraining complete in {total_time:.1f} minutes.")
print(f"Best Mean Dice: {best_metric:.4f} at Epoch {best_metric_epoch}")

## Step 8: Plot Training Loss & Validation Metrics

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(epoch_loss_values) + 1), epoch_loss_values, 'b-', label='Dice Loss', lw=2)
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
val_epochs = list(range(VAL_INTERVAL, MAX_EPOCHS + 1, VAL_INTERVAL))
plt.plot(val_epochs, metric_values, 'k-o', label='Mean Dice', lw=2)
plt.plot(val_epochs, metric_values_wt, 'g--', label='Whole Tumor (WT)')
plt.plot(val_epochs, metric_values_tc, 'r--', label='Tumor Core (TC)')
plt.plot(val_epochs, metric_values_et, 'm--', label='Enhancing Tumor (ET)')
plt.title('Validation Dice Scores')
plt.xlabel('Epoch')
plt.ylabel('Dice Score')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=300)
plt.show()

## Step 9: Visualizing Predictions vs Ground Truth (Axial Slices)

In [ ]:
# Load best model weights for visual inspection
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Pick one validation case
val_sample = val_ds[0]
sample_image = val_sample["image"].unsqueeze(0).to(device)
sample_label = val_sample["label"].numpy()

with torch.no_grad():
    val_pred = sliding_window_inference(
        inputs=sample_image,
        roi_size=ROI_SIZE,
        sw_batch_size=1,
        predictor=model,
        overlap=0.5,
    )
    val_pred = (torch.sigmoid(val_pred) > 0.5).float().cpu().numpy()[0]

# Find slice with largest tumor area
wt_slice_idx = np.argmax(np.sum(sample_label[0], axis=(0, 1)))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(sample_image.cpu().numpy()[0, 0, :, :, wt_slice_idx], cmap="gray")
axes[0].set_title(f"MRI FLAIR (Slice {wt_slice_idx})")
axes[0].axis("off")

axes[1].imshow(sample_image.cpu().numpy()[0, 1, :, :, wt_slice_idx], cmap="gray")
axes[1].set_title("MRI T1ce")
axes[1].axis("off")

axes[2].imshow(sample_label[0, :, :, wt_slice_idx], cmap="autumn", alpha=0.8)
axes[2].set_title("Ground Truth (WT)")
axes[2].axis("off")

axes[3].imshow(val_pred[0, :, :, wt_slice_idx], cmap="cool", alpha=0.8)
axes[3].set_title("AI Prediction (WT)")
axes[3].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/sample_prediction.png", dpi=300)
plt.show()

## Step 10: Export Model & Test Volume for NeuroVision 3D Web App
We save:
1. `neurovision_3d_best_model.pth`: Clean state dictionary for our FastAPI backend.
2. `sample_test_volume.npz`: One test MRI volume + predicted mask + affine matrix, so you can test the 3D Marching Cubes viewer immediately on your local machine!

In [ ]:
# Export sample data for local 3D testing
np.savez_compressed(
    "/kaggle/working/sample_test_volume.npz",
    image=sample_image.cpu().numpy()[0],
    label=sample_label,
    prediction=val_pred,
)

# Metadata for the web app
model_metadata = {
    "architecture": "SegResNet",
    "spatial_dims": 3,
    "in_channels": 4,
    "out_channels": 3,
    "channel_names": ["Whole_Tumor_WT", "Tumor_Core_TC", "Enhancing_Tumor_ET"],
    "input_modalities": ["FLAIR", "T1ce", "T1", "T2"],
    "roi_size": ROI_SIZE,
    "best_validation_dice": float(best_metric),
    "best_epoch": int(best_metric_epoch)
}

with open("/kaggle/working/neurovision_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=4)

print("✅ All deliverables exported to /kaggle/working/:")
print(" - /kaggle/working/neurovision_checkpoints/neurovision_3d_best_model.pth")
print(" - /kaggle/working/sample_test_volume.npz")
print(" - /kaggle/working/neurovision_metadata.json")
print(" - /kaggle/working/training_curves.png")
print(" - /kaggle/working/sample_prediction.png")